In [ ]:
import logging
import sys

from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue import DynamicFrame

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# Configure job-specific settings through AWS Glue job arguments.
args = getResolvedOptions(
    sys.argv,
    ['JOB_NAME', 'REDSHIFT_TMP_DIR', 'REDSHIFT_CONNECTION_NAME']
)
redshift_tmp_dir = args['REDSHIFT_TMP_DIR']
redshift_connection_name = args['REDSHIFT_CONNECTION_NAME']

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init(args['JOB_NAME'], args)
logger.info("Starting Glue Redshift load job %s", args['JOB_NAME'])

In [ ]:
# ===================================
#     Load DataFrames from Glue Catalog
# ===================================

# Each dynamic frame here represents a table in the AWS Glue Data Catalog
# These are the cleaned AdventureWorks tables getting ready to ingest into Redshift

aw_territories_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_territories", transformation_ctx="aw_territories_df"
)

aw_returns_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_returns", transformation_ctx="aw_returns_df"
)

aw_sales_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_sales", transformation_ctx="aw_sales_df"
)

aw_products_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_products", transformation_ctx="aw_products_df"
)

aw_product_category_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_product_category", transformation_ctx="aw_product_category_df"
)

aw_customers_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_customers", transformation_ctx="aw_customers_df"
)

aw_product_subcategories_df = glueContext.create_dynamic_frame.from_catalog(
    database="aw_db", table_name="aw_product_subcategories", transformation_ctx="aw_product_subcategories_df"
)



In [ ]:
# Write a DynamicFrame to the existing Redshift Glue connection.
def export_to_redshift(df, table_name, ddl, ctx_name):
    try:
        glueContext.write_dynamic_frame.from_options(
            frame=df,
            connection_type="redshift",
            connection_options={
                "redshiftTmpDir": redshift_tmp_dir,
                "useConnectionProperties": "true",
                "dbtable": f"public.{table_name}",
                "connectionName": redshift_connection_name,
                "preactions": ddl
            },
            transformation_ctx=ctx_name
        )
        logger.info("Loaded table public.%s", table_name)
    except Exception:
        logger.exception("Failed loading table public.%s", table_name)
        raise

In [ ]:
# Export each cleaned table to Redshift.
export_to_redshift(
    aw_territories_df,
    "aw_territories",
    """CREATE TABLE IF NOT EXISTS public.aw_territories (
        salesterritorykey INTEGER,
        region VARCHAR,
        country VARCHAR,
        continent VARCHAR
    );""",
    "Export_aw_territories"
)

export_to_redshift(
    aw_returns_df,
    "aw_returns",
    """CREATE TABLE IF NOT EXISTS public.aw_returns (
        returndate DATE,
        territorykey INTEGER,
        productkey INTEGER,
        returnquantity INTEGER
    );""",
    "Export_aw_returns"
)

export_to_redshift(
    aw_sales_df,
    "aw_sales",
    """CREATE TABLE IF NOT EXISTS public.aw_sales (
        orderdate DATE,
        stockdate DATE,
        ordernumber VARCHAR,
        productkey INTEGER,
        customerkey INTEGER,
        territorykey INTEGER,
        orderlineitem BIGINT,
        orderquantity BIGINT,
        sales_year INTEGER
    );""",
    "Export_aw_sales"
)

export_to_redshift(
    aw_products_df,
    "aw_products",
    """CREATE TABLE IF NOT EXISTS public.aw_products (
        productkey INTEGER,
        productsubcategorykey INTEGER,
        productsku VARCHAR,
        productname VARCHAR,
        modelname VARCHAR,
        productdescription VARCHAR,
        productcolor VARCHAR,
        productsize VARCHAR,
        productstyle VARCHAR,
        productcost DOUBLE PRECISION,
        productprice DOUBLE PRECISION
    );""",
    "Export_aw_products"
)

export_to_redshift(
    aw_product_category_df,
    "aw_product_category",
    """CREATE TABLE IF NOT EXISTS public.aw_product_category (
        productcategorykey INTEGER,
        categoryname VARCHAR
    );""",
    "Export_aw_product_category"
)

export_to_redshift(
    aw_customers_df,
    "aw_customers",
    """CREATE TABLE IF NOT EXISTS public.aw_customers (
        customerkey INTEGER,
        birthdate DATE,
        maritalstatus VARCHAR,
        gender VARCHAR,
        emailaddress VARCHAR,
        annualincome INTEGER,
        totalchildren INTEGER,
        educationlevel VARCHAR,
        occupation VARCHAR,
        homeowner VARCHAR,
        customername VARCHAR
    );""",
    "Export_aw_customers"
)

export_to_redshift(
    aw_product_subcategories_df,
    "aw_product_subcategories",
    """CREATE TABLE IF NOT EXISTS public.aw_product_subcategories (
        productsubcategorykey INTEGER,
        subcategoryname VARCHAR,
        productcategorykey INTEGER
    );""",
    "Export_aw_product_subcategories"
)

job.commit()
logger.info("Glue Redshift load job %s completed successfully", args['JOB_NAME'])